In [ ]:
import clickhouse_connect
import pandas as pd
import os
import uuid

pd.options.display.max_colwidth = 100

# Вбейте свой телеграм никнейм, чтобы в случае проблем мы могли вас индефицировать
database = 'sid998800'

client = clickhouse_connect.get_client(host='clickhouse01', port=8123, username=os.getenv('CLICKHOUSE_USER'), password=os.getenv('CLICKHOUSE_PASSWORD'))

In [ ]:
# Создаем БД
client.command(f'''
    CREATE DATABASE IF NOT EXISTS {database} ON CLUSTER '{{cluster}}';
''')


In [ ]:
client.command(f'''
    DROP TABLE IF EXISTS {database}.type_data;
''')

client.command(f'''
    CREATE TABLE {database}.type_data
    (
    --------------------------
    -- основные типы данных --
    --------------------------
        i Int8,                               -- Int8-256 (со знаком)
        ui UInt8,                             -- UInt8-256 (без знака)
        fl Float32,                           -- Float32-64 (для мат.расчетов, но не для финансов)  
        dc Decimal(9, 4),                      -- Decimal32-256 (точность после запятой)
        st String,                            -- имеет произвольную длинну
        fst FixedString(5),                   -- имеет фиксированную длинну
    --------------------------------
    -- дополнитеьлные типы данных --
    --------------------------------
        UID UUID,                             -- уникальный идентификатор
        ip4 IPv4,                             -- 127.0.0.1
        ip6 IPv6,                             -- f2c6:e19b:da60:52ad:2cef:62fe:0279
    --------------------------------
    --    типы даты и времени     --
    --------------------------------
        dt Date,                              -- Date32 (различаются диапозном дат)
        dtm DateTime,                         -- Сохрняет время с точностью то секунд
        dtm64 DateTime64,                     -- Сохрняет время с точностью то наносекунд
    ------------------------------------
    --    композитные типы данных             -- позволяют хранить более сложные структуры данных
    ------------------------------------
        ar Array(UInt8),                      -- массив данных
        tu Tuple(Date, UInt16, Decimal32(2)), -- кортеж
        ns Nested(                            -- Вложенные структуры
            col1 String,
            col2 UInt64,
            col3 String
            ),
    mp Map(String, Int16),                    -- хранит в данные в виде ключ -> значение
    en Enum('bad' = 2,                        -- хранит данные определенного значения
            'udovlet' = 3, 
            'good' = 4 )      
    )
    ENGINE = Log;
''')

# Вставляем данные в таблицу
client.command(f'''
    INSERT INTO {database}.type_data
    (
               i, ui, fl, dc, st, fst,
        UID, ip4, ip6,
        dt, dtm, dtm64,
        ar, tu, 
        ns.col1, ns.col2, ns.col3,
        mp, en
    )
    VALUES 
    (
        100,
        200,
        3.14,
        toDecimal32(3.14, 4),
        'Пример строки',
        'ABCDE',
        generateUUIDv4(),
        '192.168.1.1',
        '2001:db8::1',
        toDate('2025-04-30'),
        toDateTime('2025-04-30 14:30:00'),
        toDateTime64('2025-04-30 14:30:00.123456', 6),
        [10, 20, 30],
        (toDate('2025-04-30'), 150, 99.99),
        ['one'],
        [123456],
        ['value1'],
        {{'key1': 10, 'key2': -20}},
        'udovlet'
    );
''')

In [ ]:
# # Считываем даные из таблицы Читаем с микросекундной точностью как timestamp
# client.query_df(f'''
#     SELECT 
#         * EXCEPT(dtm64),
#         toUnixTimestamp64Micro(dtm64) * 1000 as dtm64_ns  # Наносекунды
#     FROM {database}.type_data
# ''')

Обращение к композитрым типам данных

In [ ]:
client.query_df(f'''
    SELECT 
        ar[1],      -- обращение к эл-там массива
        ar.size0,   -- получение размера массива
        tu,         -- чтение кортежа
        ns.col1,    -- обращение к вложенной структуре
        mp['key1'], -- получение данных из Map
        en          -- Чтение Enum
    FROM {database}.type_data
''')

### Задание на самостоятельную работу

1) Необходимо содать таблицу заказов (Мы еще не проходили создание таблиц поэтому создай по примеру выше с движком `Log`)

| Поле         | Описание                            |
|--------------|-------------------------------------|
| `order_id`   | Уникальный ID заказа                |
| `user_id`    | ID пользователя                     |
| `order_date` | Дата и время оформления заказа      |
| `total_amount` | Сумма заказа                      |
| `paid`       |  Признак оплаты: оплачено, не оплачено|
| `items`      |  Список ID товаров в заказе         |

2) Тебе необходимо выбрать самому подходящие типы данных для колонок
3) После создания вставь подходящие строки через `INSERT INTO <имя БД>.<имя таблицы> VALUES (...)`
4) Выведи выборку строк

In [ ]:
# Удаляем таблицу только если она существует
client.command(f'''
    DROP TABLE IF EXISTS {database}.orders;
''')
# Создаем таблицу orders с различны типами даных
client.command(f'''
    CREATE TABLE IF NOT EXISTS {database}.orders
    (
    order_id     UInt64,                    -- Целое число от 0 до 18446744073709551615
    user_id      UInt32,                    -- Целое число от 0 до 4294967295
    order_date   Date,                      -- Дата диапазон [1970-01-01, 2149-06-06]
    total_amount Decimal(10, 2),            -- Число с 2 знаками после запятой
    paid         Enum('оплачено' = 1,       -- Перечисляемый тип, состоящий из именованных значений.
                    'не оплачено' = 0),
    items       Array(UInt32)               -- Массив из целых чисел типа UInt32
)
ENGINE = Log
''')

# Добавляем данные о заказах в таблицу
client.command(f'''
INSERT INTO {database}.orders VALUES
    (1, 101, 2026-01-30, 55000.50, 1, [10, 12, 44]),
    (2, 102, 2026-01-31, 45000.45, 0, [11, 122, 144]),
    (3, 103, 2026-02-01, 35000.99, 1, [1, 2, 3, 4, 5]);
''')

In [ ]:
df = client.query_df(f'''
SELECT * FROM {database}.orders
''')

df.head()

### Функции привидения данных

In [ ]:
client.query_df('''
    select CAST('1'  as Int8)
''')

In [ ]:
# Тут будет ошибка
client.query_df('''
   select CAST(NULL as Int8)
''')

In [ ]:
client.query_df('''
    select '1'::Int64
''')

In [ ]:
client.command(f'''
    DROP TABLE IF EXISTS {database}.cast_type_data;
''')

client.command(f'''
CREATE TABLE {database}.cast_type_data (
    col String
)
ENGINE = Log;
''')

client.command(f'''
INSERT INTO {database}.cast_type_data values ('1'),('2'),('1a'),('-1')
''')

In [ ]:
client.query_df(f'''
    SELECT
        col,                                    -- исходная строка
        toInt64OrNull(col),                     -- попытка конвертировать в Int64, если не удается - NULL
        toInt8OrZero(col),                      -- конвертация в Int8, при ошибке - 0
        toInt8OrDefault(col, -100),             -- конвертация в Int8, при ошибке - -100
        toUInt8(-1),                            -- конвертация -1 в UInt8 (беззнаковый) (беззнаковый 8-бит: 0-255, -1 переполняется до 255)
        toUInt8(-1.1),                          -- конвертация -1.1 в UInt8 (округляется до -1 → 255)
        toUInt8(256)                            -- выход за пределы преобразует в значение по модулю диапозона (максимальное значение UInt8 = 255)
    FROM {database}.cast_type_data
''')

1) Выполни запросы в следующей ячейке.
2) Сделай так чтобы UNION ALL выполнился
3) Типы данных, которые не могут быть автоматически преобразованы системой должны быть, как у таблицы 1. В случае коализий преобразования типов данных, необходимо ставить значение NULL.

In [ ]:
client.command(f'''
    DROP TABLE IF EXISTS {database}.table_1;
''')

client.command(f'''
CREATE TABLE  {database}.table_1
(
    order_id     UInt32, 
    user_id      UInt16,  
    total_amount Float32,   
    paid         Nullable(UInt8),     
    item_count   Int8   
)
ENGINE = Log;
''')

client.command(f'''
    INSERT INTO {database}.table_1 VALUES (10001, 321, 1499.50, 1, 3), (10001,  321, 1499.50 , NULL, 3), (10001, 321, , 1, 3);
''')

client.command(f'''
    DROP TABLE IF EXISTS {database}.table_2;
''')

client.command(f'''
CREATE TABLE  {database}.table_2
(
    order_id     Int64, 
    user_id      UInt32,  
    total_amount Nullable(Decimal(5, 2)),   
    paid         UInt8,     
    item_count   String   
)
ENGINE = Log;
''')

client.command(f'''
    INSERT INTO {database}.table_2 VALUES (10001, 123456, 899.99, 0, '5'), (10001, 123456, NULL, 0, '5'), (10001, 123456, 899.99, 0, '5 тут специально вписаны буквы') ;
''')

In [ ]:
client.query_df(f'''
    SELECT order_id, user_id, total_amount, paid, item_count 
    FROM {database}.table_1
    UNION ALL
    SELECT order_id, user_id, toFloat32(total_amount), paid, toInt32OrNull(item_count) 
    FROM {database}.table_2
''')

# **Создание таблиц**

Создание таблиц в ClikHouse очень похоже на создание таблиц в других базах данных, за исключением того что в КХ обязательно нужно указывать движок базы данных. Каждый движок во своему уникален, но самый встречаемый и вообще именно для него и создавался КХ это MergeTree.

In [ ]:
# client.command(f'''
#     CREATE TABLE IF NOT EXISTS {database}.mt
#     (
#         id UInt32,
#         dt datetime
#     )
#     ENGINE = MergeTree                                                 -- обязательно нужно указывать движок
#     PRIMARY KEY(id)                                                    -- не обязательное поле по умолчанию равно ORDER BY
#     ORDER BY(id, dt)                                                   -- обязательно должны быть колонки в порядке из primary key
#     TTL dt + INTERVAL 1 MONTH DELETE;                                  -- от времени dt отсчитывается 1 месяц после чего данные удаляются (данные удаляются полсе слияния)
#         --dt + INTERVAL 1 MONTH DELETE WHERE toDayOfWeek(d) IN (6, 7); -- удаление с фильтрацией
#         --dt + INTERVAL 1 WEEK TO VOLUME 'aaa',                        -- перемещение данных в вольюм(совокупность дисков, задаваемая в конфигах)
#         --dt + INTERVAL 2 WEEK TO DISK 'bbb';                          -- перемещение данных на диск (указывается имя диска)
# ''')



### Самостоятельная работа на проверку работоспостобности TTL:

1) Создай таблицу с движком MergeTree c 2мя колонками: id Int32, dt DateTime
2) Укажи только сотировку по id. PRIMARY KEY указывать не нужно оно будет совпадать по умолчанию с ORDER BY
3) Напиши в условии TTL удаление строки по dt через 10 секунд
4) вставь данные через конструкцию `INSERT INTO <имя БД>.<имя таблицы> AS SELECT number, now() + number FROM numbers(100)` (`numbers(100)` это таблица которая генерирует колонку number со значениями от 0 до 99)
5) выполни select твоей таблицы и посмотри сколько в ней строк
6) как я уже говорил удаление происходит после слияния поэтому выполни команду ` OPTIMIZE TABLE <имя БД>.<имя таблицы> FINAL;` через 20 секунд после выполнения шага 5
7) выполни повторно шаг 5
8) теперь вопроряй 5-6 шаги пока в таблице совсем не останется данных

In [ ]:
client.command(f'''
    DROP TABLE IF EXISTS {database}.test_ttl;
''')

client.command(f'''
    CREATE TABLE {database}.test_ttl
    (
        id Int32,
        dt DateTime
    )
    ENGINE = MergeTree
    ORDER BY(id)
    TTL dt +INTERVAL 10 SECOND DELETE
''')

In [ ]:
client.command(f'''
    INSERT INTO {database}.test_ttl 
    SELECT 
        number, 
        now() + number 
    FROM numbers(100)
''')

In [ ]:
# Удаление происходит после слияния, для этого запускаем команду
client.command(f'''
    OPTIMIZE TABLE {database}.test_ttl FINAL
''')

In [ ]:
client.query_df(f'''
    SELECT * FROM {database}.test_ttl
''')

# **Описание полей при создании таблиц**

In [ ]:
client.command(f'''
    DROP TABLE IF EXISTS {database}.test_fields_without_ttl;
''')


client.command(f'''
    CREATE TABLE {database}.test_fields_without_ttl
    (
          col_default UInt64 DEFAULT 42                         -- значение по умолчанию
        , col_materialized UInt64 MATERIALIZED col_default * 33 -- к данной колонке можно обратиться только по имени
        , col_alias UInt64 ALIAS col_default + 1                -- к данной колонке можно обратиться только по имени
        , col_codec String CODEC(ZSTD(10))                      -- кодек сжатия
        , col_comment Date COMMENT 'Some comment'               -- комментарий к колонке
    )
    ENGINE = Log;
''')

In [ ]:
client.command(f'''
    INSERT INTO {database}.test_fields_without_ttl (
        col_default,
        col_codec,
        col_comment
    )
    SELECT
        number,
        'какой-то текст ' || toString(number),
        toDate(now()) + number
    FROM numbers(60);
''')

In [ ]:
df_tt = client.query_df(f'''
    SELECT * FROM {database}.test_fields_without_ttl
''')
df_tt.head(5)

А теперь посмотрим как работает TTL для колонкок

In [ ]:
client.command(f'''
    DROP TABLE IF EXISTS {database}.test_fields_with_ttl;
''')


client.command(f'''
    CREATE TABLE {database}.test_fields_with_ttl
    (
          col_default UInt64 DEFAULT 42
        , col_materialized UInt64 MATERIALIZED col_default * 33
        , col_alias UInt64 ALIAS col_default + 1
        , col_codec String CODEC(ZSTD(10))
        , col_comment Date COMMENT 'Some comment'
        , col_ttl UInt64 DEFAULT 10  TTL col_comment + INTERVAL 3 DAY
    )
    ENGINE = MergeTree()
    ORDER BY (col_default);
''')

client.command(f'''
    INSERT INTO {database}.test_fields_with_ttl (
        col_default,
        col_codec,
        col_comment,
        col_ttl
    )
        SELECT
            number,
            'какой-то текст ' ||  toString(number),
            toDate(now()) - number,
            rand(1) % 100000000
        FROM numbers(20);
''')

In [ ]:
client.query_df(f'''
    SELECT 
        col_default
      , col_materialized 
      , col_alias 
      , col_codec 
      , col_comment
      , col_ttl
    FROM {database}.test_fields_with_ttl
''')

С помощью следующей команды можно увидеть комментарий к столбцу

In [ ]:
client.query_df(f'''
    DESCRIBE TABLE {database}.test_fields_with_ttl -- здесь можно увидеть комментарий к столбцу
''')

### Самостоятельная работа на расшиненные атрибуты колонок:

1) Создайте таблицу `product_sales`, которая будет хранить информацию о продажах товаров. В таблице необходимо использовать **все указанные выше атрибуты хотя бы по одному разу**.

| Поле          | Тип данных  | Атрибуты                                                              |
|---------------|-------------|------------------------------------------------------------------------|
| `sale_id`     | `UInt64`    | `COMMENT`: "Уникальный идентификатор продажи"                        |
| `price`       | `Float32`   | `DEFAULT`: 0.0                                                        |
| `quantity`    | `UInt8`     | `DEFAULT`: 1                                                          |
| `total`       | `Float32`   | `MATERIALIZED`: `price * quantity`                                   |
| `description` | `String`    | `CODEC`: `ZSTD(1)`                                                    |
| `taxed_total` | `Float32`   | `ALIAS`: `total * 1.2`
2) Вставь пару строк данных в данную таблицу
3) Выведи данные через колонки и через `*`.
4) Попробуй вставить данные в колонки `total` и `taxed_total`. Посмотри на результат

In [ ]:
client.command(f'''
    DROP TABLE IF EXISTS {database}.product_sales
''')

client.command(f'''
    CREATE TABLE {database}.product_sales
    (
        sale_id UInt64 COMMENT 'Уникальный идентификатор продажи',
        price Decimal(10, 2) DEFAULT 0.0,
        quantity UInt8 DEFAULT 1,
        total Float32 MATERIALIZED price * quantity,
        description String CODEC(ZSTD(1)),
        taxed_total Float ALIAS total * 1.2
    )
    ENGINE = MergeTree
    ORDER BY(sale_id);
''')

client.command(f'''
    INSERT INTO {database}.product_sales 
    (sale_id, price, quantity, description)
    VALUES 
    (1, 29999.99, 1, 'покупка телевизора'),
    (2, 3000, 2, 'покупка мышки')
''')

client.query_df(f'''
    SELECT * FROM {database}.product_sales 
''')

client.query_df(f'''
    SELECT 
        sale_id
      , price 
      , quantity 
      , total 
      , description
      , taxed_total
    FROM {database}.product_sales
''')


Вставлять данные в материализованные и алиас колонки нельзя. Поверь в следующем запросе

In [ ]:
# client.command(f'''
#     INSERT INTO {database}.product_sales (sale_id, price, quantity, total, taxed_total, description) VALUES
#     (3, 7.0, 3, 21.0, 25.2, 'Покупка товара В');
# ''')
# # При выполнении произойдет ОШИБКА!!
# # Cannot insert column total, because it is MATERIALIZED column.

# **Атрибуты при создании колонок**

In [ ]:
client.command(f'''
    DROP TABLE IF EXISTS {database}.nl_lc_tabl;
''')

client.command(f'''
    CREATE TABLE {database}.nl_lc_tabl (
        a Nullable(UInt32),         -- разрешает вставку с пропуском значения.
        b LowCardinality(String),   -- ускоряет работу малокоординальных данных(часто повторяющихся)
        c UInt32
    ) ENGINE = MergeTree 
    ORDER BY tuple(); -- определяет порядок сток по порядку вставки данных
''')

client.command(f'''
    INSERT INTO {database}.nl_lc_tabl VALUES (NULL,'test' ,1); -- null вставится корректно
''')
client.command(f'''
    INSERT INTO {database}.nl_lc_tabl VALUES (1,'test2',NULL); -- null вставится как 0.
''')
client.command(f'''
    INSERT INTO {database}.nl_lc_tabl VALUES (1, NULL, 3); -- null вставляется как пустая строка
''')
client.command(f'''
    INSERT INTO {database}.nl_lc_tabl VALUES (1, 'test4', 4);
''')

In [ ]:
client.query_df(f'''
    SELECT * FROM {database}.nl_lc_tabl
''')

Сущесвует парамерт который запретит вставлять данные NULL в не Nullable колонки. При которой будут ошибки

In [ ]:
client.command(f'''
    SET input_format_null_as_default = 0; -- параметр отвечающий за вставку пустых значений. Выполняется совместно с командой на вставку
''')
# появится ошибка при вставке
client.command(f'''
    INSERT INTO {database}.nl_lc_tabl VALUES (2, 'sdasda', NULL)
''')

In [ ]:
client.query_df(f'''
    SELECT 
        toTypeName(a), 
        toTypeName(b), 
        toTypeName(c) 
    FROM {database}.nl_lc_tabl
''')

# **Партицирование**

**Партицирование** - это процесс объединения наборов данных по единому критерию, например по месяцу. Это позволяет ускорить процес считывания данных по выбранному критерию при фильтрации.
В ClickHouse существует 4 типа партицирования

## Диапозоном

In [ ]:
client.command(f'''
    DROP TABLE IF EXISTS {database}.table_range;
''')

client.command(f'''
    CREATE TABLE {database}.table_range
    (
        id UInt32,
        name String,
        created_at Date
    )
    ENGINE = MergeTree
    PARTITION BY
        CASE
            WHEN id < 10000 THEN 'range_1'
            WHEN id < 20000 THEN 'range_2'
            ELSE 'range_3'
        END
    ORDER BY id;
''')

client.command(f'''
    INSERT INTO {database}.table_range
    SELECT
        number AS id,
        concat('User_', toString(number)) AS name,
        today() AS created_at
    FROM
        numbers(53000)
''')

client.command(f'''
    OPTIMIZE TABLE {database}.table_range FINAL;
''')

In [ ]:
# посмотреть какие у таблицы партиции
client.query_df(f'''
    select 
        _part,
        count(*) 
    FROM 
        {database}.table_range 
    GROUP BY _part 
''')

## Интервалом

Чаще всего в проектах, даже не больших, встречается именно этот вид партицирования

In [ ]:
client.command(f'''
    DROP TABLE IF EXISTS {database}.table_interval;
''')

client.command(f'''
    CREATE TABLE {database}.table_interval
    (
        id UInt32,
        amount Float32,
        sale_date Date
    )
    ENGINE = MergeTree
    PARTITION BY toYYYYMM(sale_date)
    ORDER BY id;
''')

client.command(f'''
    INSERT INTO {database}.table_interval
    SELECT
        number AS id,
        rand() % 1000 AS amount,
        today() + (number % 90) AS sale_date
    FROM numbers(1000);
''')

client.command(f'''
    OPTIMIZE TABLE {database}.table_interval FINAL;
''')

In [ ]:
client.query_df(f'''
    SELECT * FROM {database}.table_interval
    LIMIT 10
''')

In [ ]:
# посмотреть какие у таблицы партиции
client.query_df(f'''
    select 
        _part,
        count(*) 
    FROM 
        {database}.table_interval 
    GROUP BY _part 
''')

## хеш-функцией

In [ ]:
client.command(f'''
    DROP TABLE IF EXISTS {database}.table_hash;
''')

client.command(f'''
    CREATE TABLE {database}.table_hash
    (
        user_id UInt64,
        event String
    )
    ENGINE = MergeTree
    PARTITION BY cityHash64(user_id) % 10
    ORDER BY user_id;
''')

client.command(f'''
    INSERT INTO {database}.table_hash
    SELECT
        number AS user_id,
        concat('event_', toString(number))
    FROM numbers(1000);
''')

client.command(f'''
    OPTIMIZE TABLE {database}.table_hash FINAL;
''')

In [ ]:
client.query_df(f'''
    SELECT * FROM {database}.table_hash
    LIMIT 10
''')

In [ ]:
# посмотреть какие у таблицы партиции
client.query_df(f'''
    select 
        _part,
        count(*) 
    FROM 
        {database}.table_hash 
    GROUP BY _part 
''')

## комбинированое 

In [ ]:
client.command(f'''
    DROP TABLE IF EXISTS {database}.table_composiste;
''')

client.command(f'''
    CREATE TABLE {database}.table_composiste
    (
        order_id UInt64,
        customer_id UInt64,
        order_date Date
    )
    ENGINE = MergeTree
    PARTITION BY (toYYYYMM(order_date), customer_id % 10)
    ORDER BY order_id;
''')

client.command(f'''
    INSERT INTO {database}.table_composiste
    SELECT
        number AS order_id,
        number % 100 AS customer_id,
        today() + (number % 90) AS order_date
    FROM numbers(1000);
''')

client.command(f'''
    OPTIMIZE TABLE {database}.table_composiste FINAL;
''')

In [239]:
client.query_df(f'''
    SELECT * FROM {database}.table_composiste
    LIMIT 10
''')

,order_id,customer_id,order_date
0,0,0,2026-02-03
1,10,10,2026-02-13
2,20,20,2026-02-23
3,90,90,2026-02-03
4,100,0,2026-02-13
5,110,10,2026-02-23
6,180,80,2026-02-03
7,190,90,2026-02-13
8,200,0,2026-02-23
9,270,70,2026-02-03


In [ ]:
# посмотреть какие у таблицы партиции
client.query_df(f'''
    select 
        _part,
        count(*) 
    FROM 
        {database}.table_composiste 
    GROUP BY _part 
''')

### Самостоятельная работа на партиционирование таблиц:
1) Создайте таблицу `customer_orders` с партиционированием по месяцу `PARTITION BY toYYYYMM(order_date)` и первичным ключом сортирвки  `ORDER BY` - `customer_id, order_date`. Поле `total` должно быть `MATERIALIZED`. 

| Поле             | Тип         | Описание                                 |
|------------------|-------------|------------------------------------------|
| `order_id`       | UInt64      | Уникальный идентификатор заказа          |
| `order_date`     | DateTime    | Дата и время заказа                      |
| `customer_id`    | UInt32      | ID клиента                               |
| `product_name`   | String      | Название товара                          |
| `quantity`       | UInt8       | Кол-во единиц                            |
| `price`          | Float32     | Цена за единицу                          |
| `total`          | Float32     | MATERIALIZED: `quantity * price`         |

2) Вставьте данные за несколько разных месяцев по  (можно использовать `now() - INTERVAL X DAY`)
4) Выведи данные на экран с помощью `*` и убедись в какой парции нахаодятся твои данные добавив скрытую колонку `_part`